Step 2: Text preprocessing
-Text cleaning
-Sentence segmentation
-Tokenization
-Lowercasing
-Lemmatization
-Stopword removal

In [17]:
import pandas as pd
import spacy  #used for natural language processing -> gives single best option
import nltk   #natural language toolkit-> gives multiple options
import re #regular expression
from nltk.corpus import stopwords

In [18]:
#Load our scrapped data
statements=pd.read_csv("../../data/raw/fomc_statements.csv")
minutes = pd.read_csv('../../data/raw/fomc_minutes.csv')
speeches = pd.read_csv('../../data/raw/fed_speeches.csv')

In [19]:
print(f"Statements: {statements.shape}")
print(f"Minutes: {minutes.shape}")
print(f"Speeches: {speeches.shape}")

Statements: (152, 4)
Minutes: (161, 4)
Speeches: (105, 5)


In [20]:
nltk.download('stopwords') #downloaded stopwords we want to remove
nltk.download('punkt') #a sentence tokenizer that helps split text into sentences

[nltk_data] Downloading package stopwords to /Users/paru/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/paru/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [21]:
# Load spacy model
nlp = spacy.load('en_core_web_sm')

# English stopwords
stop_words = set(stopwords.words('english'))

# Financial terms to keep
financial_terms = {
    'inflation', 'rate', 'unemployment', 'monetary', 'policy', 
    'interest', 'economic', 'growth', 'market', 'federal', 
    'reserve', 'committee', 'financial', 'price', 'stable',
    'employment', 'target', 'increase', 'decrease', 'raise',
    'cut', 'hike', 'tighten', 'accommodate', 'neutral'
}

# Remove financial terms from stopwords
stop_words = stop_words - financial_terms

print(f"Total stopwords: {len(stop_words)}")
print("spaCy and NLTK ready!")

Total stopwords: 198
spaCy and NLTK ready!


In [22]:
def preprocess_text(text):
    if not isinstance(text, str):
        return None
    
    # Step 1: Clean text - remove extra whitespace and special characters
    text = re.sub(r'\s+', ' ', text)  # normalize whitespace
    text = re.sub(r'[^a-zA-Z0-9\s\.\%\,]', '', text)  # keep letters, numbers, punctuation
    text = text.strip()
    
    # Step 2: Process with spaCy (tokenization + lemmatization)
    doc = nlp(text)
    
    # Step 3: Extract tokens - lemmatize, lowercase, remove stopwords
    tokens = [
        token.lemma_.lower() 
        for token in doc 
        if not token.is_punct  # remove punctuation
        and not token.is_space  # remove spaces
        and token.lemma_.lower() not in stop_words  # remove stopwords
        and len(token.text) > 2  # remove very short tokens
    ]
    
    # Step 4: Join tokens back into a string
    processed = ' '.join(tokens)
    
    return processed

# Test on one statement
sample = statements['text'][0]
print("ORIGINAL:")
print(sample[:300])
print("\nPROCESSED:")
print(preprocess_text(sample)[:300])

ORIGINAL:
Information received since the Federal Open Market Committee met in June indicates that the pace of recovery in output and employment has slowed in recent months. Household spending is increasing gradually, but remains constrained by high unemployment, modest income growth, lower housing wealth, and

PROCESSED:
information receive since federal open market committee meet june indicate pace recovery output employment slow recent month household spending increase gradually remains constrain high unemployment modest income growth low housing wealth tight credit business spending equipment software rise howeve


In [23]:
#applying it to all the documents
print("Processing statements...")
statements['processed_text'] = statements['text'].apply(preprocess_text)

print("Processing minutes...")
minutes['processed_text'] = minutes['text'].apply(preprocess_text)

print("Processing speeches...")
speeches['processed_text'] = speeches['text'].apply(preprocess_text)

print("Done!")


Processing statements...
Processing minutes...
Processing speeches...
Done!


In [24]:
#saving the processed data
statements.to_csv('../../data/preprocessed/fomc_statements_processed.csv', index=False)
minutes.to_csv('../../data/preprocessed/fomc_minutes_processed.csv', index=False)
speeches.to_csv('../../data/preprocessed/fed_speeches_processed.csv', index=False)

print("Saved!")

Saved!
